# 生成 BP_TBot 初始化模型

从 TBot 权重生成 BP_TBot，并做 forward。


环境配置：只放路径和运行环境相关设置。


In [ ]:
import os
import sys
from pathlib import Path

WORKSPACE_DIR = Path("/vla/workspace")
MY_TBOT_DIR = WORKSPACE_DIR / "my_tbot"
SRC_DIR = MY_TBOT_DIR / "src"
DA3_CODE_ROOT = MY_TBOT_DIR / "third_party/Depth-Anything-3"

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

for path in [SRC_DIR, DA3_CODE_ROOT]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print("MY_TBOT_DIR:", MY_TBOT_DIR)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))


常量配置：改这里即可切换模型和数据。


In [ ]:
from pathlib import Path

TBOT_BASE_MODEL_PATH = Path("/vla/workspace/models/tbot_base")
BP_SAVE_PATH = Path("/vla/workspace/models/bp_mytbot_init_new")
DATASET_PATH = Path("/vla/workspace/data/place_can_basket/aloha-agilex_clean_50")

QWEN3_VL_MODEL_PATH = Path("/vla/workspace/models/Qwen3-VL-2B-Instruct")
COSMOS_TOKENIZER_PATH = Path("/vla/workspace/models/Cosmos-Tokenizer-CI8x8")
DA3_MODEL_PATH = Path("/vla/workspace/models/DA3-LARGE-1.1")

DEVICE = "cuda"
DTYPE = "bfloat16"
BATCH_SIZE = 2
SAMPLE_INDEX = 1
SEED = 0

CHUNK_SIZE = 50
N_ACTION_STEPS = 50
BP_NUM_CHUNKS = 5
BP_ACTION_CHUNK_SIZE = 50
MAX_STATE_DIM = 32
MAX_ACTION_DIM = 32
ACTION_LOSS_VALID_DIM = 14

IMAGE_RESOLUTION = (224, 224)
IMAGE_DELTA_INDICES = [-15, 0, 15]
BP_CAMERA_KEYS = [
    "observation.images.image0",
    "observation.images.image1",
    "observation.images.image2",
]
ACTION_MODE = "delta"

for name, path in {
    "TBOT_BASE_MODEL_PATH": TBOT_BASE_MODEL_PATH,
    "DATASET_PATH": DATASET_PATH,
    "QWEN3_VL_MODEL_PATH": QWEN3_VL_MODEL_PATH,
    "COSMOS_TOKENIZER_PATH": COSMOS_TOKENIZER_PATH,
    "DA3_MODEL_PATH": DA3_MODEL_PATH,
    "DA3_CODE_ROOT": DA3_CODE_ROOT,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} 不存在: {path}")

print("BP_SAVE_PATH:", BP_SAVE_PATH)


导入依赖，并固定随机种子。


In [ ]:
import copy
import torch
from torch.utils.data import DataLoader

from lerobot.configs.policies import FeatureType, PolicyFeature
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptConfig, BehaviorPromptLeRobotDataset
from lerobot.datasets.factory import _configure_vision_only_dataset, resolve_delta_timestamps
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies.BP_TBot.configuration_bp_tbot import BPTBotConfig
from lerobot.policies.BP_TBot.modeling_bp_tbot import BPTBotPolicy

DEVICE = "cuda" if DEVICE == "cuda" and torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("device:", DEVICE)


构造 BP_TBot 配置，用于加载 TBot base。


In [ ]:
bp_policy_cfg = BPTBotConfig(
    input_features={
        "observation.state": PolicyFeature(type=FeatureType.STATE, shape=(MAX_STATE_DIM,)),
    },
    output_features={
        "action": PolicyFeature(type=FeatureType.ACTION, shape=(MAX_ACTION_DIM,)),
    },
    device=DEVICE,
    dtype=DTYPE,
    pretrained_path=None,
    push_to_hub=False,
    qwen3_vl_variant="qwen3_vl_28l",
    action_expert_variant="qwen3_28l",
    qwen3_vl_pretrained_path=str(QWEN3_VL_MODEL_PATH),
    cosmos_tokenizer_path_or_name=str(COSMOS_TOKENIZER_PATH),
    chunk_size=CHUNK_SIZE,
    n_action_steps=N_ACTION_STEPS,
    max_state_dim=MAX_STATE_DIM,
    max_action_dim=MAX_ACTION_DIM,
    mask_action_dim_padding_loss=True,
    action_loss_valid_dim=ACTION_LOSS_VALID_DIM,
    image_resolution=IMAGE_RESOLUTION,
    image_delta_indices=IMAGE_DELTA_INDICES,
    gradient_checkpointing=True,
    enable_3d_queries=True,
    num_3d_query_tokens=432,
    lambda_3d=0.01,
    da3_model_path_or_name=str(DA3_MODEL_PATH),
    da3_code_root=str(DA3_CODE_ROOT),
    log_da3_teacher_timing=True,
    bp_num_chunks=BP_NUM_CHUNKS,
    bp_action_chunk_size=BP_ACTION_CHUNK_SIZE,
    bp_use_action_step_embedding=True,
)
bp_policy_cfg.validate_features()

print("policy type:", bp_policy_cfg.type)
print("bp_num_chunks:", bp_policy_cfg.bp_num_chunks)
print("bp_action_chunk_size:", bp_policy_cfg.bp_action_chunk_size)


加载数据集，构造带 BP 的样本。


In [ ]:
repo_id = str(DATASET_PATH)

ds_meta = LeRobotDatasetMetadata(repo_id, root=DATASET_PATH)
delta_timestamps = resolve_delta_timestamps(bp_policy_cfg, ds_meta)

current_ds = LeRobotDataset(
    repo_id,
    root=DATASET_PATH,
    delta_timestamps=delta_timestamps,
    image_transforms=None,
)
frame_ds = LeRobotDataset(
    repo_id,
    root=DATASET_PATH,
    image_transforms=None,
)

_configure_vision_only_dataset(current_ds, bp_policy_cfg)

prompt_cfg = BehaviorPromptConfig(
    prompt_action_chunk_size=BP_ACTION_CHUNK_SIZE,
    same_episode_policy="avoid",
    seed=SEED,
    num_chunks=BP_NUM_CHUNKS,
    height=IMAGE_RESOLUTION[0],
    width=IMAGE_RESOLUTION[1],
    max_state_dim=MAX_STATE_DIM,
    max_action_dim=MAX_ACTION_DIM,
    qwen3_vl_processor_path=str(QWEN3_VL_MODEL_PATH),
    bp_camera_keys=BP_CAMERA_KEYS,
    action_mode=ACTION_MODE,
)

bp_ds = BehaviorPromptLeRobotDataset.with_default_transforms(current_ds, frame_ds, prompt_cfg)
bp_sample = bp_ds[SAMPLE_INDEX]

print("dataset frames:", len(bp_ds))
print("sample keys:", sorted(bp_sample.keys()))
print("BP keys:", sorted(bp_sample["behavior_prompt"].keys()))


组 batch，并检查 forward 需要的张量形状。


In [ ]:
bp_loader = DataLoader(bp_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
bp_batch = next(iter(bp_loader))

print("action:", tuple(bp_batch["action"].shape))
print("state:", tuple(bp_batch["observation.state"].shape))
print("current input_ids:", tuple(bp_batch["observation.input_ids"].shape))
print("BP action:", tuple(bp_batch["behavior_prompt"]["action"].shape))
print("BP state:", tuple(bp_batch["behavior_prompt"]["state"].shape))
print("BP input_ids:", tuple(bp_batch["behavior_prompt"]["input_ids"].shape))


从 TBot 权重初始化 BP_TBot。


In [ ]:
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

bp_policy = BPTBotPolicy.from_pretrained(
    str(TBOT_BASE_MODEL_PATH),
    config=bp_policy_cfg,
    strict=False,
)
bp_policy.train()

print("loaded from:", TBOT_BASE_MODEL_PATH)
print("bp_type_embedding:", tuple(bp_policy.model.bp_type_embedding.weight.shape))
print("bp_chunk_embedding:", tuple(bp_policy.model.bp_chunk_embedding.weight.shape))
print("bp_action_step_embedding:", tuple(bp_policy.model.bp_action_step_embedding.weight.shape))


保存新的 BP_TBot 初始化 checkpoint。


In [ ]:
BP_SAVE_PATH.mkdir(parents=True, exist_ok=True)
bp_policy.save_pretrained(BP_SAVE_PATH)

print("saved:", BP_SAVE_PATH)
print("files:", sorted(p.name for p in BP_SAVE_PATH.iterdir()))


重新加载保存后的 BP_TBot。


In [ ]:
bp_reload_cfg = copy.deepcopy(bp_policy_cfg)
bp_reload_cfg.pretrained_path = None
bp_reload_cfg.device = DEVICE

bp_policy_reloaded = BPTBotPolicy.from_pretrained(
    str(BP_SAVE_PATH),
    config=bp_reload_cfg,
    strict=False,
)
bp_policy_reloaded.train()

print("reloaded:", BP_SAVE_PATH)
print("bp_chunk_embedding:", tuple(bp_policy_reloaded.model.bp_chunk_embedding.weight.shape))


把 batch 移到模型设备。


In [ ]:
def move_to_device(value, device):
    if isinstance(value, torch.Tensor):
        return value.to(device)
    if isinstance(value, dict):
        return {k: move_to_device(v, device) for k, v in value.items()}
    if isinstance(value, list):
        return [move_to_device(v, device) for v in value]
    if isinstance(value, tuple):
        return tuple(move_to_device(v, device) for v in value)
    return value

bp_batch = move_to_device(bp_batch, DEVICE)
print("batch device:", bp_batch["action"].device)


执行 forward，确认 loss 可计算。


In [ ]:
torch.manual_seed(SEED + 1)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED + 1)

with torch.no_grad():
    loss, loss_dict = bp_policy_reloaded.forward(bp_batch)

print("loss:", float(loss.item()))
for key, value in loss_dict.items():
    print(f"{key}:", value)
